## Book Summarizer and Evaluator

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr


In [2]:
load_dotenv(override=True)

openai = OpenAI()

In [3]:
reader = PdfReader("book/TuesdaysWithMorrie.pdf")
book=""
for page in reader.pages:
    text = page.extract_text()
    if text:
        book += text

#print(book)

In [4]:
system_prompt = f"You are a helpful assistant that can summarize a book. \
You are given a book and your task is to summarize it and answer any question that the user asks about the book. \
Your responsibility is to represent the book faithfully as possible. \
Be professional and engaging."

system_prompt += f"\n\n## Summary:\n{book}"
system_prompt += f"With this context, please chat with the user, always staying in character as the book."

system_prompt 

'You are a helpful assistant that can summarize a book. You are given a book and your task is to summarize it and answer any question that the user asks about the book. Your responsibility is to represent the book faithfully as possible. Be professional and engaging.\n\n## Summary:\ntuesdays\twith\tMorrie\nan\told\tman,\ta\tyoung\tman,\tand\tlife\'s\tgreatest\tlessonby\tMitch\tAlbomAcknowledgments\n\tI\twould\tlike\tto\tacknowledge\tthe\tenormous\thelp\tgiven\tto\tme\tin\tcreating\tthis\nbook.\tFor\ttheir\tmemories,\ttheir\tpatience,\tand\ttheir\tguidance,\tI\twish\tto\tthank\nCharlotte,\tRob,\tand\tJonathan\tSchwartz,\tMaurie\tStein,\tCharlie\tDerber,\tGordie\nFellman,\tDavid\tSchwartz,\tRabbi\tAl\tAxelrad,\tand\tthe\tmultitude\tof\tMorrie\'s\nfriends\tand\tcolleagues.\tAlso,\tspecial\tthanks\tto\tBill\tThomas,\tmy\teditor,\tfor\nhandling\tthis\tproject\twith\tjust\tthe\tright\ttouch.\tAnd,\tas\talways,\tmy\tappreciation\nto\tDavid\tBlack,\twho\toften\tbelieves\tin\tme\tmore\tthan\tI\

In [15]:

messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": 'Summarize the book in 1000 words'}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
summary =  response.choices[0].message.content
print(summary)


**Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson** by Mitch Albom is a poignant memoir that shares the author's true story of reconnecting with his former college professor, Morrie Schwartz, who is facing terminal illness due to amyotrophic lateral sclerosis (ALS). The narrative unfolds through a series of meetings that take place every Tuesday, where the two discuss profound topics related to life, death, love, and the human experience.

Mitch Albom first met Morrie Schwartz when he was a student at Brandeis University in the 1970s. Morrie was a beloved sociology professor who taught with wisdom, compassion, and a deep understanding of human relationships. As a student, Mitch was deeply inspired by Morrie, who often introduced themes of love, work, family, and life’s purpose into his classroom discussions. However, after graduating, Mitch fell into the trap of a busy career in sports journalism, becoming preoccupied with success and materialism, and lost tou

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

#gr.ChatInterface(chat, type="messages").launch()

## Now creating an Evaluator model which will validate the results from the first model

In [7]:
from pydantic import BaseModel

class Evaluator(BaseModel):
    is_acceptable: bool
    feedback: str


In [16]:

evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is accurate and acceptable quality. \
The Agent is playing the role to summarize {book}. The Agent has been instructed to be professional and engaging. \
The Agent has been provided with context in the form of their summary {summary}."

evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."


In [17]:

def evaluator_user_prompt(reply, message, history) -> Evaluator:
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += f"Please evaluate the response, replying with whether it is acceptable and your feedback."

    return user_prompt

In [10]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [18]:
def evaluate(reply,message,history) -> Evaluator:
    messages = [{'role':'system','content':evaluator_system_prompt}] + [{'role':'user','content':evaluator_user_prompt(reply,message,history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluator)
    return response.choices[0].message.parsed

In [20]:
messages= [{'role':'system','content':evaluator_system_prompt}] + [{'role':'user','content':'What is this book about?'}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [21]:
from IPython.display import display, Markdown

display(Markdown(reply))

The book *Tuesdays with Morrie* by Mitch Albom reflects on the author's relationship with his former sociology professor, Morrie Schwartz, who is dying from amyotrophic lateral sclerosis (ALS). Through a series of meetings that take place every Tuesday, the two discuss significant life topics such as the meaning of life, love, work, aging, family, and death. Morrie shares his wisdom and personal insights, encouraging Mitch to reflect on his own life, prioritizing human connection and love over material success. Ultimately, the book explores profound themes of compassion and acceptance, emphasizing the importance of living fully and cherishing relationships as life draws to a close. It serves as a poignant reminder to appreciate the moments we have with those we love, and the lessons we learn from those relationships persist even after death. 

---

The response is acceptable. It accurately summarizes the main themes and context of the book, reflecting the essence of Morrie's teachings and the impact on Mitch's life. The response captures the emotional depth of the narrative and provides a comprehensive overview without losing focus on the primary subjects addressed in the memoir.

In [24]:
def chat(message, history):
    
    system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()